# 공격적 최적화: SpeedNorm + PerfNorm 동시 향상

## 모델 구조 분석 결과

```
EXAONE-4.0-1.2B 파라미터 분포:
├─ Embedding: 209.7M (16.4% of total params)
├─ Attention:  314M (24.6%)  - Q, K, V, O projections
└─ FFN:        755M (59.0%)  - gate, up, down projections
```

## 점수 공식
```
Score = 0.5 × PerfNorm + 0.5 × SpeedNorm
SpeedNorm = 1 - (model_time/token) / (base_time/token)
```

## 최적화 전략 (3가지 버전)

| 버전 | 핵심 전략 | 예상 효과 |
|------|----------|----------|
| **V1** | 강화 캘리브레이션 (512샘플, 1024길이) | PerfNorm ↑ |
| **V2** | group_size=64 (더 정밀한 양자화) | PerfNorm ↑↑ |
| **V3** | lm_head도 양자화 (최대 압축) | SpeedNorm ↑, PerfNorm ↓ |

---

# 1. Import

In [2]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("\nImport 완료!")

PyTorch: 2.9.1
CUDA: False

Import 완료!


# 2. 버전 선택

아래에서 원하는 버전을 선택하세요:

- **V1**: 강화 캘리브레이션 (안전한 선택)
- **V2**: 작은 group_size (정확도 우선)
- **V3**: lm_head 양자화 (속도 우선)

In [3]:
# ============================================================================
#  VERSION 선택 (V1, V2, V3 중 하나)
# ============================================================================

VERSION = "V3"  # V1, V2, V3

# ============================================================================

MODEL_ID = "./open/base_model"
OUT_DIR = "./model"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
ORIGINAL_SIZE_GB = 2.56

# 버전별 설정
CONFIGS = {
    "V1": {
        "name": "강화 캘리브레이션",
        "description": "더 많은 샘플, 더 긴 시퀀스로 양자화 품질 향상",
        "num_samples": 512,
        "max_seq_len": 1024,
        "group_size": 128,
        "ignore": ["embed_tokens", "lm_head"],
        "actorder": "weight",
    },
    "V2": {
        "name": "정밀 양자화 (group_size=64)",
        "description": "더 작은 group_size로 정확도 향상 (크기 약간 증가)",
        "num_samples": 512,
        "max_seq_len": 1024,
        "group_size": 64,  # 128 -> 64 (더 정밀)
        "ignore": ["embed_tokens", "lm_head"],
        "actorder": "weight",
    },
    "V3": {
        "name": "최대 압축 (lm_head 포함)",
        "description": "lm_head도 양자화하여 속도 향상",
        "num_samples": 512,
        "max_seq_len": 1024,
        "group_size": 128,
        "ignore": ["embed_tokens"],  # lm_head 양자화!
        "actorder": "weight",
    },
}

config = CONFIGS[VERSION]

print("=" * 60)
print(f"선택: {VERSION} - {config['name']}")
print(f"설명: {config['description']}")
print("=" * 60)
print(f"  캘리브레이션: {config['num_samples']}샘플, {config['max_seq_len']}길이")
print(f"  group_size: {config['group_size']}")
print(f"  ignore: {config['ignore']}")
print(f"  actorder: {config['actorder']}")
print("=" * 60)

선택: V3 - 최대 압축 (lm_head 포함)
설명: lm_head도 양자화하여 속도 향상
  캘리브레이션: 512샘플, 1024길이
  group_size: 128
  ignore: ['embed_tokens']
  actorder: weight


# 3. 모델 로드

In [4]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

print(f"[INFO] 파라미터: {model.num_parameters():,}")
print("[INFO] 완료")

[INFO] 모델 로드 중...


`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 파라미터: 1,279,391,488
[INFO] 완료


# 4. 데이터셋 로드

In [5]:
NUM_SAMPLES = config['num_samples']
MAX_SEQ_LEN = config['max_seq_len']

print(f"[INFO] 캘리브레이션 데이터: {NUM_SAMPLES}개, {MAX_SEQ_LEN} 길이")

ds = load_dataset(
    DATASET_ID,
    split=f"train[:{NUM_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)
print(f"[INFO] 준비 완료: {len(ds)}개")

[INFO] 캘리브레이션 데이터: 512개, 1024 길이
[INFO] 준비 완료: 512개


# 5. 양자화 실행

In [6]:
print("=" * 60)
print(f"양자화: {VERSION} - {config['name']}")
print("=" * 60)

recipe = [
    GPTQModifier(
        scheme="W4A16",
        targets=["Linear"],
        ignore=config['ignore'],
        block_size=config['group_size'],
        dampening_frac=0.001,
        actorder=config['actorder'],
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQ_LEN,
    num_calibration_samples=NUM_SAMPLES,
)

print("\n양자화 완료!")

양자화: V3 - 최대 압축 (lm_head 포함)


Tokenizing:   0%|          | 0/512 [00:00<?, ? examples/s]

2026-02-11T17:45:06.366229+0900 | reset | INFO - Compression lifecycle reset
2026-02-11T17:45:06.367588+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-11T17:45:06.387727+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-11T17:45:06.388129+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-11T17:45:06.393256+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0211 17:45:06.417000 15586 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [05:34<00:00,  1.53it/s]

2026-02-11T17:50:41.372065+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 512 samples


...

(19/31): Calibrating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [05:34<00:00,  1.53it/s]

2026-02-11T21:00:24.677090+0900 | compress_modules | INFO - Quantizing model.layers.18.self_attn.q_proj using 512 samples


2026-02-11T21:00:24.977485+0900 | compress | METRIC - time 0.30s
2026-02-11T21:00:24.977970+0900 | compress | METRIC - error 946.43
2026-02-11T21:00:24.978749+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-11T21:00:24.978941+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-11T21:00:24.980581+0900 | compress_modules | INFO - Quantizing model.layers.18.self_attn.k_proj using 512 samples
2026-02-11T21:00:25.176497+0900 | compress | METRIC - time 0.20s
2026-02-11T21:00:25.176960+0900 | compress | METRIC - error 269.72
2026-02-11T21:00:25.177725+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-11T21:00:25.177948+0900 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-11T21:00:25.178712+0900 | compress_modules | INFO - Quantizing model.layers.18.self_attn.v_proj using 512 samples
2026-02-11T21:00:25.365967+0900 | compress | METRIC - time 0.19s
2026-02-11T21:00:

(20/31): Calibrating:  48%|█████████████████████████████████████████████████████████████████████████▏                                                                               | 245/512 [02:43<02:58,  1.50it/s]


KeyboardInterrupt: 

# 6. 모델 저장

In [ ]:
print("[INFO] 모델 저장...")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 크기 확인
total_size = sum(
    os.path.getsize(os.path.join(OUT_DIR, f)) 
    for f in os.listdir(OUT_DIR)
)
quantized_gb = total_size / 1e9

print("\n" + "=" * 60)
print(f"{VERSION}: {config['name']}")
print("=" * 60)
print(f"원본: {ORIGINAL_SIZE_GB:.2f} GB")
print(f"압축: {quantized_gb:.2f} GB ({quantized_gb/ORIGINAL_SIZE_GB*100:.1f}%)")
print("=" * 60)

# 7. 제출 파일 생성

In [ ]:
zip_name = f"submit_{VERSION.lower()}"

if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(zip_name, "zip", ".", OUT_DIR)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9

print("=" * 60)
print(f"제출 파일: {zip_name}.zip")
print(f"크기: {zip_size:.2f} GB (제한: 10GB)")
print("=" * 60)

---

# 예상 결과 비교

| 버전 | 설정 | 크기 | PerfNorm | SpeedNorm | Score |
|------|------|------|----------|-----------|-------|
| 베이스라인 | 256샘플, gs=128 | 1.42GB | ~0.95 | ~0.30 | ~0.625 |
| **V1** | 512샘플, 1024길이 | 1.42GB | **~0.96** | ~0.30 | **~0.63** |
| **V2** | group_size=64 | 1.6GB | **~0.97** | ~0.25 | ~0.61 |
| **V3** | lm_head 양자화 | **1.0GB** | ~0.93 | **~0.45** | **~0.69** |

## 권장 순서

1. **V1** 먼저 시도 (안전 + 개선 가능성)
2. 결과 확인 후 **V3** 시도 (속도 우선이면)
3. 성능 중요하면 **V2** 시도

---

## 추가 고려사항

### lm_head 양자화 (V3) 주의점
- `tie_word_embeddings=true`이므로 embed_tokens와 공유
- lm_head 양자화 시 출력 품질에 영향
- 하지만 vocab_size가 102400으로 커서 크기 효과 큼

### group_size 효과
- group_size=64: 더 정밀하지만 크기 증가
- group_size=128: Marlin 최적화 호환
- group_size=256: 덜 정밀하지만 크기 감소

---